# CV Masterclass 07: Metric Learning & Face Recognition

Welcome to Notebook 07. If you are building a facial recognition system for an airport, you cannot use a standard ImageNet ResNet classifier with a final Softmax layer.

---

## 🎓 Socratic Deep Check
Ponder these questions before we begin:

> *"Why does a standard Softmax classifier completely fail in a Biometric Security system where a newly hired employee needs to be added to the database tomorrow?"*

> *"ArcFace achieves state-of-the-art face recognition by enforcing large angular margins on a hypersphere. But at inference time, the ArcFace head (the weight matrix W and the angular margin) is discarded entirely — only the CNN backbone is used to extract features. This means the hypersphere geometry was ONLY needed during training. Why does enforcing a geometric constraint during training (with tools you discard at inference) result in better deployable representations than just training with standard cross-entropy loss?"*

---

## Table of Contents
1. **Closed-Set vs Open-Set:** The Softmax bottleneck.
2. **Triplet Loss & Online Hard Negative Mining:** The Anchor, Positive, and Negative paradigm.
3. **Margin-based Softmax:** CosFace, ArcFace and the hypersphere geometry.
4. **Production Retrieval (FAISS):** Using FAISS for sub-millisecond vector retrieval.


## 1. Closed-Set vs Open-Set

**Closed-Set Classification:** (e.g., ImageNet or MNIST). You have 1,000 fixed classes. You train the network to output a vector of length 1,000. You apply Softmax. You pick the highest probability. 
*   **The Problem:** If you are Apple building FaceID, you don't know the faces of the 100 Million users who will buy the iPhone next year. You cannot train a Softmax layer with 100 Million outputs.

**Open-Set Classification (Metric Learning):** Instead of learning a *classification*, you train the CNN to map an image of a face into a 512-dimensional vector. 
You mathematically enforce a geometry: If two images are of the same person, their 512-dimensional vectors must be extremely close together (Euclidean Distance or Cosine Similarity). If they are different people, the vectors must be far apart.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# Demonstration of the Softmax Bottleneck:
# A basic class-based bottleneck model where dimensions equal the number of classes.
class ClosedSetModel(nn.Module):
    def __init__(self, num_classes=1000):
        super().__init__()
        self.fc = nn.Linear(512, num_classes)
        
    def forward(self, x):
        return F.softmax(self.fc(x), dim=1)

# TEST IT
x = torch.randn(2, 512)
model = ClosedSetModel(num_classes=1000)
out = model(x)
print(f"Closed-Set Output Shape: {out.shape}")  # (2, 1000)
# We simply don't have a 100-million way softmax.


### COMMON PITFALLS
*   **Assuming Softmax Extrapolates:** Believing the final layer features before softmax will automatically act as good open-set distance embeddings. Without metric-based regularization, standard cross-entropy features cluster loosely and lack intra-class compactness.
*   **Closed-Set Evaluation:** Evaluating an open-set identity model using standard classification accuracy on a test set, instead of retrieval metrics (like Recall@K) or verification metrics (like TAR@FAR).


## 2. Triplet Loss

The original Google FaceNet (2015) used **Triplet Loss** to enforce this geometry.

You feed the network 3 images simultaneously:
1.  **Anchor ($A$):** A picture of Employee John.
2.  **Positive ($P$):** A completely different picture of Employee John.
3.  **Negative ($N$):** A picture of Employee Sarah.

**The Math (Step-by-Step Derivation):**

1.  We want the distance between Anchor and Positive to be small: $d(A, P) \to 0$
2.  We want the distance between Anchor and Negative to be large: $d(A, N) \to \infty$
3.  Instead of absolute values, we enforce a *relative* margin $\alpha$. The negative should be further from the anchor than the positive by at least $\alpha$:
    $$d(A, P) + \alpha < d(A, N)$$
4.  Rearranging to form a loss function where values $\le 0$ mean zero loss:
    $$d(A, P) - d(A, N) + \alpha \le 0$$
5.  Using squared L2 distance and a max function to clamp at 0:
    $$L = \max(||f(A) - f(P)||^2 - ||f(A) - f(N)||^2 + \alpha, 0)$$

### The Mining Problem: Online Hard Negative Mining

With $N$ identities and $P$ photos per identity in a dataset, the number of possible triplets is $O(N^3 \cdot P^3)$. You cannot possibly enumerate or train on all triplets.

**Types of Negatives:**
*   **Easy negatives:** $d(A, N) \gg d(A, P)$. The negative is already far away. These produce near-zero loss. Training with them stagnates because gradients are zero.
*   **Hard negatives:** $d(A, N) < d(A, P)$. The negative is closer to the anchor than the positive is. Produces high loss, but if you only mine hard negatives early in training, the model collapses by violently updating weights based on outliers or mislabeled data.
*   **Semi-hard negatives:** $d(A, P) < d(A, N) < d(A, P) + \alpha$. This is the sweet spot. The negative is correctly further than the positive, but still inside the margin boundary. Google FaceNet explicitly targets semi-hard negatives.

Let's implement online semi-hard mining for a batch without any $O(N^3)$ outer loops—using purely matrix broadcasting. If we have a batch size of $B$ with $P$ photos per person, computing the pairwise distances is just matrix multiplication!
For a batch of 128 samples with $P=4$ photos per person, there are 32 identities.
The number of valid triplets = $\text{Identities} \times C(Photos, 2) \times \text{Negatives}$.
$32 \times C(4,2) \times 124 = 32 \times 6 \times 124 = 23,808$ possible triplets in just one batch of 128!


In [ ]:
def pairwise_distances(embeddings):
    # compute ||x - y||^2 = ||x||^2 - 2<x,y> + ||y||^2
    dot_product = torch.matmul(embeddings, embeddings.t())
    square_norm = torch.diag(dot_product)
    # Expand dims for broadcasting
    distances = square_norm.unsqueeze(0) - 2.0 * dot_product + square_norm.unsqueeze(1)
    # Clamp to avoid negative distances due to numerical precision
    distances = torch.clamp(distances, min=0.0)
    return distances

def semi_hard_triplet_loss(embeddings, labels, margin=0.2):
    # 1. Pairwise distance matrix
    dist_mat = pairwise_distances(embeddings)
    
    # 2. Masking to find valid anchor-positive and anchor-negative pairs
    label_eq = (labels.unsqueeze(0) == labels.unsqueeze(1))
    identity_mask = torch.eye(labels.size(0), dtype=torch.bool, device=labels.device)
    pos_mask = label_eq & ~identity_mask
    neg_mask = ~label_eq
    
    # 3. For each anchor, get hardest positive (max pos distance)
    # We replace invalid pos distances with 0 so they aren't chosen
    pos_dists = dist_mat * pos_mask.float()
    hardest_pos_dists, _ = pos_dists.max(dim=1, keepdim=True)
    
    # 4. For each anchor, get hardest negative (min neg distance > 0)
    max_dist = dist_mat.max().item()
    neg_dists = dist_mat + (~neg_mask).float() * (max_dist + 1e5)
    hardest_neg_dists, _ = neg_dists.min(dim=1, keepdim=True)
    
    # 5. Triplet loss calculation
    loss = torch.clamp(hardest_pos_dists - hardest_neg_dists + margin, min=0.0)
    return loss.mean()

# TEST IT
# Batch of 8 samples: 4 identities, 2 photos each. Embeddings are 16 dims.
torch.manual_seed(42)
test_embeddings = torch.randn(8, 16)
# L2 Normalize
test_embeddings = F.normalize(test_embeddings, p=2, dim=1)
test_labels = torch.tensor([0, 0, 1, 1, 2, 2, 3, 3])

loss_val = semi_hard_triplet_loss(test_embeddings, test_labels, margin=0.2)
print(f"Batch loss: {loss_val.item():.4f}")
print(f"Pairwise dist matrix shape: {pairwise_distances(test_embeddings).shape}") # (8, 8)


### COMMON PITFALLS
*   **Offline Mining Trap:** Attempting to build a massive list of triplets across the entire dataset before training. It becomes rapidly stale since embeddings change every step. Always mine *online* within the batch.
*   **Batch Size Too Small:** If your batch size is 16 or 32, there might be no valid semi-hard negatives inside the batch. Metric learning usually requires large batch sizes (e.g. 512+) or careful sampler designs (PK sampler: P identities, K images per identity).
*   **Collapsing to Zero:** If the network outputs all zeros or a constant vector, the distance between any A, P, N is 0, so $d(A,P) - d(A,N) + \alpha = \alpha$. To avoid degeneracies, embeddings are strictly L2 normalized.


## 3. Margin-based Softmax

Triplet Loss works but it's tricky to tune (the combinatorial explosion and mining logic).

Modern face recognition returns to the beloved Softmax equation but hacks the underlying dot products.

**Standard Softmax Derivation:**
The denominator uses the dot product between weights and features: $W^T x$. 
We know mathematically that:
$$W_j^T x_i = ||W_j|| \cdot ||x_i|| \cdot \cos(\theta_{j,i})$$

By mathematically forcing the class Weights ($W$) and the Features ($x$) to have a magnitude of exactly $1.0$ (L2 Normalization), the dot product strictly becomes $\cos(\theta)$—the angle between the feature and the class center on a perfect hypersphere.

### CosFace (Large Margin Cosine Loss)
CosFace acts as the conceptual bridge between standard Softmax and ArcFace.
The standard normalized softmax target probability is proportional to $e^{s \cdot \cos(\theta)}$.
CosFace enforces margin in **cosine space** by subtracting a margin $m$:
$$P = e^{s \cdot (\cos(\theta_{y_i}) - m)}$$

*Why subtract?* It makes the target logit artificially *smaller*. To get an equivalent softmax probability, the network must squeeze the $\theta$ between the feature and the class center to be much tighter. The margin is linear in cosine distance.

### ArcFace (Additive Angular Margin Loss)
ArcFace enforces margin in **angular space**:
$$P = e^{s \cdot \cos(\theta_{y_i} + m)}$$

*Key Difference:* CosFace's margin $m$ acts on the cosine value, so its geometric effect varies depending on where you are on the sphere. ArcFace's margin acts directly on the angle, meaning the decision boundary maintains constant angular separation (arc-length) across the entire hypersphere geometry. This often makes ArcFace more robust and easier to converge.


In [ ]:
class CosFace(nn.Module):
    def __init__(self, in_features, out_classes, s=64.0, m=0.35):
        super(CosFace, self).__init__()
        self.s = s
        self.m = m
        self.weight = nn.Parameter(torch.FloatTensor(out_classes, in_features))
        nn.init.xavier_uniform_(self.weight)

    def forward(self, x, labels):
        # 1. L2 Normalize features and weights
        cosine = F.linear(F.normalize(x), F.normalize(self.weight))
        
        # 2. Add margin penalty in Cosine space (ONLY for true class)
        target_logits = cosine - self.m
        
        # 3. One-hot recombine
        one_hot = torch.zeros_like(cosine)
        one_hot.scatter_(1, labels.view(-1, 1).long(), 1)
        output = (one_hot * target_logits) + ((1.0 - one_hot) * cosine)
        
        # 4. Scale
        return output * self.s

class ArcFace(nn.Module):
    def __init__(self, in_features, out_classes, s=64.0, m=0.50):
        super(ArcFace, self).__init__()
        self.s = s
        self.m = m
        self.weight = nn.Parameter(torch.FloatTensor(out_classes, in_features))
        nn.init.xavier_uniform_(self.weight)

    def forward(self, x, labels):
        cosine = F.linear(F.normalize(x), F.normalize(self.weight))
        
        # Safe acos
        theta = torch.acos(torch.clamp(cosine, -1.0 + 1e-7, 1.0 - 1e-7))
        
        # Add margin penalty in Angular space
        target_logits = torch.cos(theta + self.m)
        
        one_hot = torch.zeros_like(cosine)
        one_hot.scatter_(1, labels.view(-1, 1).long(), 1)
        output = (one_hot * target_logits) + ((1.0 - one_hot) * cosine)
        
        return output * self.s

# TEST IT
b_size, f_dim, num_cls = 4, 128, 10
features = torch.randn(b_size, f_dim)
labels = torch.tensor([1, 4, 1, 9])

cosface_criterion = CosFace(f_dim, num_cls)
arcface_criterion = ArcFace(f_dim, num_cls)

cos_out = cosface_criterion(features, labels)
arc_out = arcface_criterion(features, labels)

print(f"CosFace Output Shape: {cos_out.shape} (-m applied appropriately)")
print(f"ArcFace Output Shape: {arc_out.shape} (+m applied appropriately)")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Comparison Plot: Standard Softmax vs CosFace vs ArcFace bounds
theta_vals = np.linspace(0, np.pi, 200)

margin_cos = 0.35
margin_arc = 0.50

# Logits before scaling
standard_logits = np.cos(theta_vals)
cosface_logits = np.cos(theta_vals) - margin_cos
arcface_logits = np.cos(theta_vals + margin_arc)

plt.figure(figsize=(8, 5))
plt.plot(np.degrees(theta_vals), standard_logits, label='Standard Softmax', lw=2)
plt.plot(np.degrees(theta_vals), cosface_logits, label=f'CosFace (m={margin_cos})', lw=2)
plt.plot(np.degrees(theta_vals), arcface_logits, label=f'ArcFace (m={margin_arc})', lw=2)

plt.axhline(0, color='black', linewidth=0.8, linestyle='--')
plt.title("Target Logit Penalties over Angle ($\theta$)")
plt.xlabel("Angle $\theta$ (degrees)")
plt.ylabel("Target Logit Value")
plt.legend()
plt.grid(True, alpha=0.3)
plt.xlim(0, 180)
plt.tight_layout()
plt.show()

# TEST IT: The plot will generate visually showing ArcFace falls to -1 much faster (steeper penalty).


### COMMON PITFALLS
*   **Forgetting the Scale Parameter $s$:** If you just output $\cos(\theta)$, your logits are trapped between $-1$ and $1$. Softmax gradients would vanish rapidly. The hyperparameter $s$ scales the hypersphere radius (typically $s=64$ for face recognition) to restore proper loss gradients.
*   **Not using L2 Normalization at Inference:** If you use an ArcFace backbone, its raw outputs *must* be L2 normalized before calculating cosine similarities or Euclidean distances at inference time.


## 4. Production Retrieval (FAISS)

Once you have an ArcFace/CosFace model, you deploy it.

1.  New employee is hired $\to$ Camera takes photo $\to$ CNN extracts 512-dim vector $\to$ Saved to DB.
2.  Employee walks up to security gate $\to$ Camera extracts live 512-dim vector.

If your company has 100,000 employees, you must calculate the Cosine Distance (+ threshold) between the live vector and all 100,000 vectors instantly.
A naive `for` loop or standard SQL queries will be catastrophic for latency. Instead, we use **FAISS (Facebook AI Similarity Search)**. FAISS excels at sub-millisecond retrieval on billions of vectors.

Let's implement the full FAISS retrieval pipeline showing three core indices:
1.  **IndexFlatIP:** Exact Inner Product Search. Since our vectors are L2 normalized, the inner product perfectly equals the cosine similarity. The search is exact (100% recall), but scales linearly $O(N)$.
2.  **IndexIVFFlat:** Inverted File with Flat Index. We cluster the dataset into Voronoi cells using K-Means (centroids). At query time, we find the closest $k$ centroids and only search those buckets.
3.  **IndexIVFPQ:** Inverted File with Product Quantization. We further compress the vectors within buckets by slicing them into sub-vectors and indexing them, requiring drastically less RAM.

*The `nprobe` parameter controls how many centroid buckets we search at query time, dictating the latency vs recall tradeoff.*


In [ ]:
import faiss
import time

d = 512          # Embedding dimension
nb = 100000      # 100k synthetic embeddings in database
nq = 100         # 100 live queries to test

# 1. Create a synthetic dataset of 100,000 512-dim face embeddings
np.random.seed(42)
db_vectors = np.random.randn(nb, d).astype('float32')
faiss.normalize_L2(db_vectors)  # ArcFace normalized!

query_vectors = np.random.randn(nq, d).astype('float32')
faiss.normalize_L2(query_vectors)

# Create indices
nlist = 256 # Number of Voronoi cells
m = 16      # Sub-quantizers for PQ (512 must be a multiple of m)

# 2. IndexFlatIP (Exact Search)
index_flat = faiss.IndexFlatIP(d)

# 3. IndexIVFFlat
quantizer = faiss.IndexFlatIP(d)
index_ivf = faiss.IndexIVFFlat(quantizer, d, nlist, faiss.METRIC_INNER_PRODUCT)

# 4. IndexIVFPQ
index_pq = faiss.IndexIVFPQ(quantizer, d, nlist, m, 8) 
# Note: we don't pass METRIC_INNER_PRODUCT explicitly depending on FAISS version, PQ defaults to L2 or depends on quantizer

print(f"{'Index Type':<15} | {'Build/Train':<12} | {'Query (100)':<12} | {'Recall@1'}")
print("-" * 55)

# --- Benchmark FlatIP ---
t0 = time.time()
index_flat.add(db_vectors)
build_time_flat = time.time() - t0

t0 = time.time()
D_flat, I_flat = index_flat.search(query_vectors, 1)
query_time_flat = time.time() - t0

print(f"{'FlatIP':<15} | {build_time_flat:<12.4f} | {query_time_flat:<12.4f} | {'100.0%':<8}")

# --- Benchmark IVFFlat ---
t0 = time.time()
index_ivf.train(db_vectors)    # K-Means clustering
index_ivf.add(db_vectors)
build_time_ivf = time.time() - t0

index_ivf.nprobe = 10  # Search 10 out of 256 lists
t0 = time.time()
D_ivf, I_ivf = index_ivf.search(query_vectors, 1)
query_time_ivf = time.time() - t0

# Calculate Recall@1 vs Exact search
recall_ivf = np.sum(I_ivf[:, 0] == I_flat[:, 0]) / nq * 100

print(f"{'IVFFlat':<15} | {build_time_ivf:<12.4f} | {query_time_ivf:<12.4f} | {f'{recall_ivf:.1f}%':<8}")

# --- Benchmark IVFPQ ---
t0 = time.time()
index_pq.train(db_vectors)
index_pq.add(db_vectors)
build_time_pq = time.time() - t0

index_pq.nprobe = 10
t0 = time.time()
D_pq, I_pq = index_pq.search(query_vectors, 1)
query_time_pq = time.time() - t0

recall_pq = np.sum(I_pq[:, 0] == I_flat[:, 0]) / nq * 100
print(f"{'IVFPQ':<15} | {build_time_pq:<12.4f} | {query_time_pq:<12.4f} | {f'{recall_pq:.1f}%':<8}")

# TEST IT: The code runs, indexes 100,000 vectors, and performs fast similarity searches correctly.


### COMMON PITFALLS
*   **Adding before Training:** For IVF or PQ indexes, you MUST call `index.train()` before calling `index.add()`. If training data is smaller than your whole DB, use a representative subset.
*   **Not tuning `nprobe`:** Leaving `nprobe=1` (the default) often leads to terrible recall on IVF indexes. Tuning `nprobe` allows you to directly slide the dial between latency and accuracy.
*   **Blindly using L2 distance with Cosine features:** If your models emit Cosine similarities, `IndexFlatL2` works but distances are inverted (smaller is better). Using `IndexFlatIP` directly calculates Cosine similarity dynamically if the embeddings are pre-normalized, removing confusion.
